# Photogrammetric Optode Co-Registration - NEMES 2026 workshop

This is adapted from Cedalion tutorial notebooks. For details on each step, see Cedalion docs:
- [Tutorial 2 – Photogrammetric Optode Co-Registration](https://doc.ibs.tu-berlin.de/cedalion/doc/dev/examples/tutorial/2_photogrammetry.html)

Note about 3D scanners:

The Cedalion tutorial notebooks and some functions assume you are using a [Shining3D Einstar](https://www.einstar.com/products/einstar-2). These structured-light scanners will have a higher precision than using a smartphone or similar. However, we can still produce a mesh with a smartphone using an app like [Scaniverse](https://dev.scaniverse.com), or Structure-from-Motion libraries like [COLMAP](https://colmap.github.io/index.html), although these will be less precise. There are also other scanners like the [Structure](https://structure.io/) sensors that can produce detailed meshes but anything other than the Einstar used in the official Cedalion notebooks ideally need some validation.

In this workshop we will be working with a Scaniverse mesh for simplicity's sake.


In [ ]:
import glob
import os

import cedalion
import cedalion.io
import cedalion.io.snirf
import cedalion.nirs.cw
import cedalion.vis.anatomy
import cedalion.vis.blocks as vbx

import cedalion.dataclasses as cdc

import matplotlib.pyplot as plt
import pyvista as pv
import numpy as np
import xarray as xr

from cedalion import units
from cedalion.geometry.photogrammetry.processors import (
    ColoredStickerProcessor,
    geo3d_from_scan,
)
from cedalion.vis.anatomy import OptodeSelector
from cedalion.geometry.registration import find_spread_points
from cedalion.dataclasses.geometry import PointType
from pathlib import Path

np.set_printoptions(suppress=True)
pv.set_jupyter_backend("server")

# What counts as a short channel?
DIST_THRESHOLD = 1.5 * units.cm 

# File locations
snirf_file = "../data/example.snirf"
example_scan = "../data/example_scan/scan.obj"

# output location
output_dir = Path("../data/registration")
output_dir.mkdir(parents=True, exist_ok=True)

# Load the mesh

We can load a mesh in `obj` format using the `read_einstar_obj`. This functions expects and `.obj`, with an associated material `.mtl` and image texture such as `texture.jpg`.

Note that einstar convention assumes the mesh is in `mm` units, while Scaniverse and other programs will produce other units. This is noted in the [function docstring](https://github.com/ibs-lab/cedalion/blob/dev/src/cedalion/io/probe_geometry.py):

> Units of the vertex coordinates in the file. Defaults to millimetres (Einstar convention). Pass e.g. ``cedalion.units.m`` for Scaniverse exports.

The extent (size) of a mesh produced by an Einstar of a head might be around `[300,300,300]`. This size matters because the toolbox functions make some assumptions about the size of identified color circles for identifying optodes. Therefore you might have to scale up your mesh.


In [ ]:
# Load our scan
surface_mesh = cedalion.io.read_einstar_obj(example_scan, units=cedalion.units.m)
display(surface_mesh)

# Check the extents
print(surface_mesh.mesh.extents)

# Convert to mm, which is einstar convention
mesh_mm = surface_mesh.mesh.copy()
mesh_mm.apply_scale(1000.0)
surface_mesh_mm = cdc.TrimeshSurface(mesh_mm, crs=surface_mesh.crs, units=cedalion.units.mm)

# Identify optode positions

The `ColoredStickerProcessor` can identify circular areas of a specified colors on the mesh and assign tentative optode positions to those.

Since it will not always get it correct, the toolbox also offers a manual selection to clean up the designated optodes.

In [ ]:
# This will process colored circles for automatic optode detection.
# Standard is yellow; 0.11, 0.21, 0.7, 1.0
# Adapt it to the colors picked up by the scan.
processor = ColoredStickerProcessor(
    colors={
        "O" : ((0.11, 0.21, 0.6, 1.0)), # (hue_min, hue_max, value_min, value_max)
    },
    cluster_eps=0.5
)

# Select the hue and value ranges based on this preview of the distribution of
# vertices in color space.
preview_details = processor.inspect_colors(surface_mesh_mm)
preview_details.plot_vertex_colors()

In [ ]:
# process the mesh
sticker_centers, normals, details = processor.process(surface_mesh_mm, details=True)
display(sticker_centers)

# Manual selection

The `OptodeSelector` is a convenient tool that displays the mesh and allows you to add/delete optodes by right clicking in the 3D view.

In [ ]:
optode_selector = OptodeSelector(surface_mesh_mm, sticker_centers, normals)
optode_selector.plot()
optode_selector.enable_picking()
vbx.plot_surface(optode_selector.plotter, surface_mesh_mm, opacity=1.0)

optode_selector.plotter.show()

In [ ]:
sticker_centers = optode_selector.points.copy()
normals = optode_selector.normals.copy()
display(sticker_centers)

# Get scalp coords

We need the length of the optode holders to project from sticker center down onto scalp.

In [ ]:
# The height of the optode holder (or rather, from scalp to sticker).
optode_length = 22.6 * cedalion.units.mm

# Calculate scalp position from sticker centers and length.
scalp_coords = sticker_centers.copy()
mask_optodes = sticker_centers.group == "O"
scalp_coords[mask_optodes] = (
    sticker_centers[mask_optodes] - optode_length * normals[mask_optodes]
)

# we make a copy of this raw set of scalp coordinates to use later in the 2nd case of
# the coregistration example that showcases an alternative route if landmark-based
# coregistration fails
scalp_coords_altcase = scalp_coords.copy()

display(scalp_coords)

In [ ]:
# Check the projection from sticker to scalp.
pvplt = pv.Plotter()
vbx.plot_surface(pvplt, surface_mesh_mm, opacity=0.3)
vbx.plot_labeled_points(pvplt, sticker_centers, color="r")
vbx.plot_labeled_points(pvplt, scalp_coords, color="g")
vbx.plot_vector_field(pvplt, sticker_centers, normals)
pvplt.show()

# Identify landmarks

If we did not automatically identify landmarks with the sticker processor, we can select them here.

In [ ]:
# Display the selector
pvplt = pv.Plotter()
get_landmarks = vbx.plot_surface(pvplt, surface_mesh_mm, opacity=1.0, pick_landmarks=True)
pvplt.show()

In [ ]:
# Get the landmarks we selected in previous step
landmarks = get_landmarks()
display(landmarks)
assert len(set(landmarks.label.values)) == 5, "please select 5 landmarks"

# Map the scanned geometry onto the probe in the SNIRF file

First we load our original SNIRF file and make sure landmark locations are named the same as our mesh.

In [ ]:
rec = cedalion.io.snirf.read_snirf(snirf_file)[0]
print(rec)

amp = rec.get_timeseries()
print("Amplitude shape:", dict(amp.sizes))
print("Wavelengths (nm):", amp.wavelength.values)

# Plot montage
cedalion.vis.anatomy.montage.plot_montage3D(rec['amp'], rec.geo3d)

In [ ]:
# read 3D coordinates of the optodes
montage_elements = rec.geo3d

print("Geo3d labels:", montage_elements.label.values)

# landmark labels must match exactly. Adjust case where they don't match.
montage_elements = montage_elements.points.rename({"LPA": "Lpa", 
                                                   "RPA": "Rpa",
                                                   "NASION": "Nz"})

In [ ]:
# This is some extra short channel handling outside the scope of the tutorial notebooks.
# Since we have not scanned short channels on the inside of the cap, we will handle them separately.

# Channels shorter than threshold are treated as short-separation channels for this montage.
ts_long, ts_short = cedalion.nirs.split_long_short_channels(
    rec["amp"], montage_elements, distance_threshold=DIST_THRESHOLD
)
long_labels = set(ts_long.source.values) | set(ts_long.detector.values)
short_labels = set(ts_short.source.values) | set(ts_short.detector.values)

# Optodes that are *only* used by short channels: photogrammetry of the cap
# surface would typically fail to detect these.
hidden_labels = sorted(short_labels - long_labels)
print(f"{len(hidden_labels)} optodes are only used by short channels: {hidden_labels}")

# Keep the complete versions around; we restore them later.
amp_full = rec["amp"]
montage_elements_full = montage_elements

# Continue without short channels.
rec["amp"] = ts_long
montage_elements = montage_elements.drop_sel(label=hidden_labels)

# Transform coordinate system

The montage from the SNIRF file and the mesh are in different coordinate systems.

Display how we will transform from one coordinate system to the other, via landmarks.

In [ ]:
f = plt.figure(figsize=(12,5))
ax1 = f.add_subplot(1,2,1, projection="3d")
ax2 = f.add_subplot(1,2,2, projection="3d")
colors = {cdc.PointType.SOURCE: "r", cdc.PointType.DETECTOR: "b"}
sizes = {cdc.PointType.SOURCE: 20, cdc.PointType.DETECTOR: 20}

for i, (type, x) in enumerate(montage_elements.groupby("type")):
    x = x.pint.to("mm").pint.dequantify()
    ax1.scatter(x[:, 0], x[:, 1], x[:, 2], c=colors.get(type, "g"), s=sizes.get(type, 2))

for i, (type, x) in enumerate(landmarks.groupby("type")):
    x = x.pint.to("mm").pint.dequantify()
    ax2.scatter(x[:, 0], x[:, 1], x[:, 2], c=colors.get(type, "g"), s=20)

for ax, points in [(ax1, montage_elements), (ax2, landmarks)]:
    points = points.pint.to("mm").pint.dequantify()
    ax.plot([points.loc["Nz",0], points.loc["Iz",0]],
            [points.loc["Nz",1], points.loc["Iz",1]],
            [points.loc["Nz",2], points.loc["Iz",2]],
            c="k"
            )
    ax.plot([points.loc["Lpa",0], points.loc["Rpa",0]],
            [points.loc["Lpa",1], points.loc["Rpa",1]],
            [points.loc["Lpa",2], points.loc["Rpa",2]],
            c="k"
            )

ax1.set_title(f"from snirf | crs: {montage_elements.points.crs}")
ax2.set_title(f"from scan | crs: {landmarks.points.crs}");

# Perform the transformation

Using landmarks, we apply the transformation between montage and mesh.

In [ ]:
trafo = cedalion.geometry.registration.register_trans_rot(landmarks, montage_elements)

filtered_montage_elements = montage_elements.where(
    (montage_elements.type == cdc.PointType.SOURCE)
    | (montage_elements.type == cdc.PointType.DETECTOR),
    drop=True,
)
filtered_montage_elements_t = filtered_montage_elements.points.apply_transform(trafo)

pvplt = pv.Plotter()
vbx.plot_surface(pvplt, surface_mesh_mm, color="w", opacity=.2)
vbx.plot_labeled_points(pvplt, filtered_montage_elements_t)
pvplt.show()

# Map each montage optode onto registered mesh optodes.

In [ ]:
# iterative closest point registration
idx = cedalion.geometry.registration.icp_with_full_transform(
    scalp_coords, filtered_montage_elements_t, max_iterations=100
)

# extract labels for detected optodes
label_dict = {}
for i, label in enumerate(filtered_montage_elements.coords["label"].values):
    label_dict[i] = label
labels = [label_dict[index] for index in idx]

# write labels to scalp_coords
scalp_coords = scalp_coords.assign_coords(label=labels)

# add landmarks
geo3Dscan = geo3d_from_scan(scalp_coords, landmarks)

display(geo3Dscan)

# Compare original montage and registered montage

In [ ]:
f,ax = plt.subplots(1,2, figsize=(12,6))
cedalion.vis.anatomy.scalp_plot(
    rec["amp"],
    montage_elements,
    cedalion.nirs.channel_distances(rec["amp"], montage_elements),
    ax=ax[0],
    optode_labels=True,
    cb_label="channel dist. / mm",
    cmap="plasma",
    vmin=25,
    vmax=42,
)
ax[0].set_title("montage from snirf file")
cedalion.vis.anatomy.scalp_plot(
    rec["amp"],
    geo3Dscan,
    cedalion.nirs.channel_distances(rec["amp"], geo3Dscan),
    ax=ax[1],
    optode_labels=True,
    cb_label="channel dist. / mm",
    cmap="plasma",
    vmin=25,
    vmax=42,
)
ax[1].set_title("montage from photogrammetric scan")
plt.tight_layout()

# Add short channels back

This step can be skipped if there are not short channels in the montage, or if you are able to scan your short channels properly.

In [ ]:
# Which optodes have we already added?
common_labels = [
    l for l in geo3Dscan.label.values if l in montage_elements_full.label.values
]

# Transform the short channels using the previous transformation matrix
missing_estimated = montage_elements_full.sel(
    label=hidden_labels
).points.apply_transform(trafo)

# Add the transformed short channels to the mesh
geo3Dscan = xr.concat([geo3Dscan.drop_vars("group", errors="ignore"), missing_estimated], dim="label")
rec["amp"] = amp_full  # restore the full channel list
display(geo3Dscan.sel(label=hidden_labels))

In [ ]:
# Some extra validation in case an optode got assigned as a landmark or vice versa.

# Are label types correct?
landmarks_found = geo3Dscan.sel(
    label=geo3Dscan.type == cedalion.dataclasses.geometry.PointType.LANDMARK
)
print(landmarks_found.label.values)

# Make sure S and D are sources/detectors.
types = geo3Dscan.type.values.copy()
for i, lbl in enumerate(geo3Dscan.label.values):
    if lbl.startswith("S"):
        types[i] = PointType.SOURCE
    elif lbl.startswith("D"):
        types[i] = PointType.DETECTOR

# Assign correct types to mesh
geo3Dscan = geo3Dscan.assign_coords(
    type=("label", types)
)

# What are the landmarks in our mesh now?
landmarks_found = geo3Dscan.sel(
    label=geo3Dscan.type == cedalion.dataclasses.geometry.PointType.LANDMARK
)
print(landmarks_found.label.values)

# Save final geometry

We will export the registered optodes and landmarks to a `.tsv` file that can be read back and applied in downstream processing.

In [ ]:
# Export all points (optodes + landmarks)
tsv_filename = output_dir / "geometry.tsv"

cedalion.io.probe_geometry.export_to_tsv(
    tsv_filename,
    geo3Dscan,
)